In [1]:
import copy
import datetime as dt
from datetime import datetime
import importlib  # needed so that we can reload packages
import logging
import os
import pathlib
import sys
import time
import warnings
from typing import Union, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# --- Make notebooks/ importable from any sector subfolder ---
import os, sys, pathlib
_NB_PARENT = pathlib.Path(os.getcwd()).parent  # notebooks/<sector>/ -> notebooks/
if str(_NB_PARENT) not in sys.path:
    sys.path.insert(0, str(_NB_PARENT))

from utils.logger_utils import setup_clean_logger, mute_external_loggers

# SISEPUEDE imports
from sisepuede.manager.sisepuede_examples import SISEPUEDEExamples
from sisepuede.manager.sisepuede_file_structure import SISEPUEDEFileStructure
import sisepuede.core.support_classes as sc
import sisepuede.utilities._plotting as spu
import sisepuede.utilities._toolbox as sf
import sisepuede.core.attribute_table as att
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
import sisepuede.visualization.plots as svp



# --- Runtime configuration ---
warnings.filterwarnings("ignore")

# Set up a clean logger for your notebook
logger = setup_clean_logger("notebook", logging.INFO)
logger.info("Notebook started successfully.")

# Mute logs from sisepuede to avoid duplication
mute_external_loggers(["sisepuede"])
# run this on the terminal /opt/anaconda3/envs/ssp_libya_env_2/bin/python -m pip install "setuptools>=60,<70"

2026-09-17 17:27:13,928 - INFO - Notebook started successfully.


In [2]:
%load_ext autoreload
%autoreload 2

### Initial Set up

Make sure to edit the config yaml under ssp_modeling/config_files/config.yaml

You can also create a new config yaml



In [3]:
# Set up dir paths

CURR_DIR_PATH = pathlib.Path(os.getcwd())
NOTEBOOKS_DIR_PATH = CURR_DIR_PATH.parent  # notebooks/<sector>/ -> notebooks/
SSP_MODELING_DIR_PATH = NOTEBOOKS_DIR_PATH.parent
PROJECT_DIR_PATH = SSP_MODELING_DIR_PATH.parent
DATA_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("input_data")
RUN_OUTPUT_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("ssp_run_output")
SCENARIO_MAPPING_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("scenario_mapping")
CONFIG_DIR_PATH = NOTEBOOKS_DIR_PATH.joinpath("config_files")
TRANSFORMATIONS_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("transformations")
MISC_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("misc")
STRATEGIES_DEFINITIONS_FILE_PATH = TRANSFORMATIONS_DIR_PATH.joinpath("strategy_definitions.csv")
STRATEGY_MAPPING_FILE_PATH = MISC_DIR_PATH.joinpath("strategy_mapping.yaml")

In [4]:
from ssp_transformations_handler.GeneralUtils import GeneralUtils
from ssp_transformations_handler.TransformationUtils import TransformationYamlProcessor, StrategyCSVHandler

# Initialize general utilities
g_utils = GeneralUtils()

In [5]:
# Load config file, double check your parameters are correct

YAML_FILE_PATH = os.path.join(CONFIG_DIR_PATH, "config.yaml")
config_params = g_utils.read_yaml(YAML_FILE_PATH)

country_name = config_params['country_name']
ssp_input_file_name = config_params['ssp_input_file_name']
ssp_transformation_cw = config_params['ssp_transformation_cw']
energy_model_flag = config_params['energy_model_flag']
set_lndu_reallocation_factor_to_zero_flag = config_params['set_lndu_reallocation_factor_to_zero']
sim_end_year = config_params.get('sim_end_year', 2050)  # Default to 2050 if not specified

# Print config parameters
logger.info(f"Country name: {country_name}")
logger.info(f"SSP input file name: {ssp_input_file_name}")
logger.info(f"SSP transformation CW: {ssp_transformation_cw}")
logger.info(f"Energy model flag: {energy_model_flag}")
logger.info(f"Set lndu reallocation factor to zero flag: {set_lndu_reallocation_factor_to_zero_flag}")
logger.info(f"Simulation end year: {sim_end_year}")

2026-09-17 17:27:14,022 - INFO - Country name: morocco
2026-09-17 17:27:14,022 - INFO - SSP input file name: sisepuede_raw_input_morocco_fuels_inen_ippu_scoe_prodcal_BLD_DEMAND_UP.csv
2026-09-17 17:27:14,023 - INFO - SSP transformation CW: ssp_morocco_transformation_cw.xlsx
2026-09-17 17:27:14,023 - INFO - Energy model flag: True
2026-09-17 17:27:14,023 - INFO - Set lndu reallocation factor to zero flag: True
2026-09-17 17:27:14,023 - INFO - Simulation end year: 2050


In [6]:
def get_file_structure(
    y0: int = 2015,
    y1: int = sim_end_year,
) -> Tuple[sfs.SISEPUEDEFileStructure, att.AttributeTable]:
    """Get the SISEPUEDE File Structure and update the attribute table
        with new years.
    """
    # setup some SISEPUEDE variables and update time period
    file_struct = sfs.SISEPUEDEFileStructure(
        initialize_directories = False,
    )
 
    # get some keys
    key_time_period = file_struct.model_attributes.dim_time_period
    key_year = file_struct.model_attributes.field_dim_year
 
 
    ##  BUILD THE ATTRIBUTE AND UPDATE
 
    # setup the new attribute table
    years = np.arange(y0, y1 + 1, ).astype(int)
    attribute_time_period = att.AttributeTable(
        pd.DataFrame(
            {
                key_time_period: range(len(years)),
                key_year: years,
            }
        ),
        key_time_period,
    )
 
    # finally, update the ModelAttributes inside the file structure
    (
        file_struct
        .model_attributes
        .update_dimensional_attribute_table(
            attribute_time_period,
        )
    )
 
    # return the tuple
    out = (file_struct, attribute_time_period, )
 
    return out


In [7]:
# Set up SSP objects
INPUT_FILE_PATH = DATA_DIR_PATH.joinpath(ssp_input_file_name)

# model attributes and associated support classes
_EXAMPLES = sxl.SISEPUEDEExamples()
_FILE_STRUCTURE, _ATTRIBUTE_TABLE_TIME_PERIOD = get_file_structure(y1=sim_end_year)
matt = _FILE_STRUCTURE.model_attributes
regions = sc.Regions(matt, )
time_periods = sc.TimePeriods(matt, )
 

In [8]:
# setup models in case we need them
models = sm.SISEPUEDEModels(
    matt,
    allow_electricity_run = True,
    fp_julia = _FILE_STRUCTURE.dir_jl,
    fp_nemomod_reference_files = _FILE_STRUCTURE.dir_ref_nemo,
    initialize_julia = False, 
)

### Making sure our input file has the correct format and correct columns
We use an example df with the complete fields and correct format to make sure our file is in the right shape

In [9]:
##  BUILD BASE INPUTS
df_inputs_raw = pd.read_csv(INPUT_FILE_PATH)

# pull example data to fill in gaps
df_example_input = _EXAMPLES("input_data_frame")

In [10]:
# Double checking that our df is in the correct shape (Empty sets should be printed to make sure everything is Ok!)
g_utils.compare_dfs(df_example_input, df_inputs_raw)

Columns in df_example but not in df_input: {'frac_trns_fuelmix_water_borne_gasoline', 'frac_lndu_seasonal_wetland_shrublands', 'frac_lndu_seasonal_wetland_croplands', 'frac_lndu_seasonal_wetland_pastures', 'frac_lndu_seasonal_wetland_other', 'region', 'frac_frst_forest_conversions_available_for_fuelwood', 'frac_lndu_seasonal_wetland_grasslands'}
Columns in df_input but not in df_example: {'ef_fgtv_distribution_tonne_co2_per_m3_fuel_crude', 'frac_fgtv_capture_associated_gas_fuel_oil', 'frac_fgtv_capture_associated_gas_fuel_coal', 'ef_fgtv_distribution_tonne_ch4_per_m3_fuel_crude', 'iso_alpha_3', 'year', 'frac_fgtv_capture_associated_gas_fuel_crude'}


In [11]:
all_fields = matt.all_variable_fields_input
df_fiels = df_inputs_raw.columns.tolist()
missing_fields = list(set(all_fields) - set(df_fiels))
print("Missing fields in the input data frame:")
print(missing_fields)

Missing fields in the input data frame:
['frac_trns_fuelmix_water_borne_gasoline', 'frac_lndu_seasonal_wetland_shrublands', 'frac_lndu_seasonal_wetland_croplands', 'frac_lndu_seasonal_wetland_pastures', 'frac_lndu_seasonal_wetland_other', 'frac_frst_forest_conversions_available_for_fuelwood', 'frac_lndu_seasonal_wetland_grasslands']


In [12]:
# Ensure if time_period field exist
if 'time_period' not in df_inputs_raw.columns:
    logger.info("Adding 'time_period' column to df_inputs_raw")
    df_inputs_raw = df_inputs_raw.rename(columns={'period':'time_period'})
else:
    logger.info("'time_period' column already exists in df_inputs_raw")

2026-09-17 17:27:14,778 - INFO - 'time_period' column already exists in df_inputs_raw


In [13]:
# Fixes differences and makes sure that our df is in the correct format.
# Note: Edit this if you need more changes in your df

df_inputs_raw_complete = g_utils.add_missing_cols(df_example_input, df_inputs_raw.copy())
df_inputs_raw_complete.head()

,time_period,year,area_gnrl_country_ha,area_lndu_infimum_croplands_ha,area_lndu_infimum_flooded_ha,area_lndu_infimum_forests_mangroves_ha,area_lndu_infimum_forests_primary_ha,area_lndu_infimum_forests_secondary_ha,area_lndu_infimum_grasslands_ha,area_lndu_infimum_other_ha,...,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha,region,frac_trns_fuelmix_water_borne_gasoline,frac_frst_forest_conversions_available_for_fuelwood,frac_lndu_seasonal_wetland_croplands,frac_lndu_seasonal_wetland_grasslands,frac_lndu_seasonal_wetland_other,frac_lndu_seasonal_wetland_pastures,frac_lndu_seasonal_wetland_shrublands
0,0,2015,44655000,-999,-999,-999,-999,-999,-999,-999,...,24.760000,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2016,44655000,-999,-999,-999,-999,-999,-999,-999,...,25.570737,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,2017,44655000,-999,-999,-999,-999,-999,-999,-999,...,24.673359,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2018,44655000,-999,-999,-999,-999,-999,-999,-999,...,25.327768,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,2019,44655000,-999,-999,-999,-999,-999,-999,-999,...,28.151940,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# Double checking that our df is in the correct shape (Empty sets should be printed to make sure everything is Ok!)
g_utils.compare_dfs(df_example_input, df_inputs_raw_complete)

Columns in df_example but not in df_input: set()
Columns in df_input but not in df_example: {'ef_fgtv_distribution_tonne_co2_per_m3_fuel_crude', 'frac_fgtv_capture_associated_gas_fuel_oil', 'frac_fgtv_capture_associated_gas_fuel_coal', 'ef_fgtv_distribution_tonne_ch4_per_m3_fuel_crude', 'iso_alpha_3', 'year', 'frac_fgtv_capture_associated_gas_fuel_crude'}


In [15]:
df_inputs_raw_complete["region"].head()

0    costa_rica
1    costa_rica
2    costa_rica
3    costa_rica
4    costa_rica
Name: region, dtype: object

In [16]:
# Set region to country name
df_inputs_raw_complete['region'] = country_name
df_inputs_raw_complete['region'].head()

0    morocco
1    morocco
2    morocco
3    morocco
4    morocco
Name: region, dtype: object

In [17]:
# filter to match sim_end_year
print(f"min and max years in raw inputs before filtering: {df_inputs_raw_complete['year'].min()} to {df_inputs_raw_complete['year'].max()}")
df_inputs_raw_complete = df_inputs_raw_complete[df_inputs_raw_complete['year'] <= sim_end_year]
print(f"min and max years in raw inputs after filtering: {df_inputs_raw_complete['year'].min()} to {df_inputs_raw_complete['year'].max()}")

min and max years in raw inputs before filtering: 2015 to 2070
min and max years in raw inputs after filtering: 2015 to 2050


In [18]:
df_inputs_raw_complete[['ef_frst_sequestration_primary_kt_co2_ha','ef_frst_sequestration_secondary_kt_co2_ha','ef_frst_sequestration_young_secondary_kt_co2_ha']].head()

,ef_frst_sequestration_primary_kt_co2_ha,ef_frst_sequestration_secondary_kt_co2_ha,ef_frst_sequestration_young_secondary_kt_co2_ha
0,0.000025,0.000053,0.000103
1,0.000025,0.000053,0.000103
2,0.000025,0.000053,0.000103
3,0.000025,0.000053,0.000103
4,0.000025,0.000053,0.000103


In [19]:
df_inputs_raw_complete['frac_frst_forest_conversions_available_for_fuelwood']

0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
20    0.0
21    0.0
22    0.0
23    0.0
24    0.0
25    0.0
26    0.0
27    0.0
28    0.0
29    0.0
30    0.0
31    0.0
32    0.0
33    0.0
34    0.0
35    0.0
Name: frac_frst_forest_conversions_available_for_fuelwood, dtype: float64

### Custom data modifications (in-memory on `df_inputs_raw_complete`)

Adjustments applied directly to the input dataframe BEFORE building the strategies. Each modification is documented with its rationale and IEA reference. **These are not changes to `df_input_1.csv` on disk** — they live in this notebook so the data source remains reproducible.

| Change | Variable | Value | Reason | Reference |
|---|---|---:|---|---|
| Add gas MSP for BAU | `nemomod_entc_frac_min_share_production_pp_gas` | 0.10 (all years) | df_input_1 has gas MSP = 0, so NemoMod never dispatches natural gas in BAU. Forcing 10% aligns with Morocco 2018 IEA actual (~10-12% gas in generation mix). | IEA Energy Statistics Morocco 2018-2023; ONEE annual reports |


In [20]:
# CUSTOM DATA MODIFICATIONS
# Apply gas MSP = 0.10 so BAU dispatches natural gas to match IEA Morocco mix
# (df_input_1.csv has gas MSP = 0, which causes NemoMod to never dispatch gas)

import numpy as np
_gas_msp_col = 'nemomod_entc_frac_min_share_production_pp_gas'
_gas_msp_target = 0.10

assert _gas_msp_col in df_inputs_raw_complete.columns, f'Column {_gas_msp_col} missing'
_before = df_inputs_raw_complete[_gas_msp_col].iloc[0]
df_inputs_raw_complete[_gas_msp_col] = _gas_msp_target
_after = df_inputs_raw_complete[_gas_msp_col].iloc[0]
print(f'gas MSP: {_before} -> {_after} (applied to all {len(df_inputs_raw_complete)} rows)')

# Verify total MSP per year stays below 1.0 (NemoMod needs headroom)
_mix_cols = [c for c in df_inputs_raw_complete.columns if c.startswith('nemomod_entc_frac_min_share_production_pp_')]
_sums = df_inputs_raw_complete[_mix_cols].sum(axis=1)
print(f'Sum of MSPs per year: min={_sums.min():.3f}, max={_sums.max():.3f}')
assert _sums.max() <= 1.0, 'MSP sum > 1.0 — would cause LP infeasibility'

# Show resulting MSP composition (non-zero only)
print('\nMSP composition after modification (sample at 2018):')
_row = df_inputs_raw_complete[df_inputs_raw_complete['time_period']==3].iloc[0]
for c in sorted(_mix_cols):
    v = _row[c]
    if v > 0:
        tech = c.replace('nemomod_entc_frac_min_share_production_pp_', '')
        print(f'  {tech:<20} {v:.4f}')
print(f'  {"SUM":<20} {sum(_row[c] for c in _mix_cols):.4f}')

gas MSP: 0.1 -> 0.1 (applied to all 36 rows)
Sum of MSPs per year: min=0.990, max=0.990

MSP composition after modification (sample at 2018):
  coal                 0.6200
  gas                  0.1000
  hydropower           0.0200
  nuclear              0.0100
  oil                  0.0500
  solar                0.0400
  wind                 0.1500
  SUM                  0.9900


#  Let's try building transformations using this


In [21]:
import sisepuede.transformers as trf
transformers = trf.transformers.Transformers(
    {},
    attr_time_period = _ATTRIBUTE_TABLE_TIME_PERIOD,
    df_input = df_inputs_raw_complete, # here you change the input table
)

##  Instantiate some transformations. Make sure to run this cell to create the transformations folder for the first time or if you wish to overwrite

In [22]:
# set an ouput path and instantiate
if not TRANSFORMATIONS_DIR_PATH.exists():
    trf.instantiate_default_strategy_directory(
        transformers,
        TRANSFORMATIONS_DIR_PATH,
    )
else:
    logger.info(f"Directory {TRANSFORMATIONS_DIR_PATH} already exists. Skipping instantiation.")


2026-09-17 17:27:15,262 - INFO - Directory /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/transformations already exists. Skipping instantiation.


In [23]:
# then, you can load this back in after modifying (play around with it)
transformations = trf.Transformations(
    TRANSFORMATIONS_DIR_PATH,
    transformers = transformers,
)
tab = transformations.attribute_transformation.table

In [24]:
#  build the strategies -- will export to path
t0 = time.time()
strategies = trf.Strategies(
    transformations,
    export_path = "transformations",
    prebuild = True,
)

t_elapse = sf.get_time_elapsed(t0)
print(f"Strategies defined at {strategies.transformations.dir_init} initialized in {t_elapse} seconds")

Strategies defined at /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/transformations initialized in 26.52 seconds


In [25]:
strategies.attribute_table

,strategy_id,strategy_code,strategy,description,transformation_specification,baseline_strategy_id
0,0,BASE,Strategy TX:BASE,NaN,TX:BASE,1
1,1000,AGRC:DEC_CH4_RICE,Singleton - Default Value - AGRC: Improve rice...,NaN,TX:AGRC:DEC_CH4_RICE,0
2,1001,AGRC:DEC_EXPORTS,Singleton - Default Value - AGRC: Decrease Exp...,NaN,TX:AGRC:DEC_EXPORTS,0
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN,Singleton - Default Value - AGRC: Reduce suppl...,NaN,TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN,0
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE,Singleton - Default Value - AGRC: Expand conse...,NaN,TX:AGRC:INC_CONSERVATION_AGRICULTURE,0
...,...,...,...,...,...,...
153,6084,WHIRLPOOL_PFLO_LEDS:TX:TRWW:INC_CAPTURE_BIOGAS...,Remove TX:TRWW:INC_CAPTURE_BIOGAS_STRATEGY_LED...,NaN,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0
154,6085,WHIRLPOOL_PFLO_LEDS:TX:TRWW:INC_COMPLIANCE_SEP...,Remove TX:TRWW:INC_COMPLIANCE_SEPTIC_STRATEGY_...,NaN,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0
155,6086,WHIRLPOOL_PFLO_LEDS:TX:WALI:INC_TREATMENT_INDU...,Remove TX:WALI:INC_TREATMENT_INDUSTRIAL_STRATE...,NaN,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0
156,6087,WHIRLPOOL_PFLO_LEDS:TX:WASO:DEC_CONSUMER_FOOD_...,Remove TX:WASO:DEC_CONSUMER_FOOD_WASTE_STRATEG...,NaN,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0


In [26]:
strategies_to_run = [0, 6004]
strategies_to_run.extend(range(6047, 6089))
print(f"Strategies to run: {strategies_to_run}")

Strategies to run: [0, 6004, 6047, 6048, 6049, 6050, 6051, 6052, 6053, 6054, 6055, 6056, 6057, 6058, 6059, 6060, 6061, 6062, 6063, 6064, 6065, 6066, 6067, 6068, 6069, 6070, 6071, 6072, 6073, 6074, 6075, 6076, 6077, 6078, 6079, 6080, 6081, 6082, 6083, 6084, 6085, 6086, 6087, 6088]


##  Build our templates
- let's use the default variable groupings for LHS

In [27]:
# Building excel templates, make sure to include the strategies ids in the strategies attribute as well as the baseline (0)
df_vargroups = _EXAMPLES("variable_trajectory_group_specification")

strategies.build_strategies_to_templates(
    # df_trajgroup = df_vargroups,
    # include_simplex_group_as_trajgroup = True,
    strategies = strategies_to_run,
)

0

# Finally, load SISEPUEDE and run it

In [28]:
import sisepuede as si
# timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
ssp = si.SISEPUEDE(
    "calibrated",
    db_type = "csv",
    # id_str = f"sisepuede_run_2024-11-04T09:23:26.721580",
    initialize_as_dummy = not(energy_model_flag), # no connection to Julia is initialized if set to True
    regions = [country_name],
    strategies = strategies,
    # try_exogenous_xl_types_in_variable_specification = True,
    attribute_time_period=_ATTRIBUTE_TABLE_TIME_PERIOD
)

2026-09-17 17:28:54,668 - INFO - Successfully initialized SISEPUEDEFileStructure.
2026-09-17 17:28:54,669 - WARNING - Missing key dict_dimensional_keys: key time_series not found. Tables that rely on the time_series will not have index checking.
2026-09-17 17:28:54,670 - INFO - 	Setting export engine to 'csv'.
2026-09-17 17:28:54,670 - WARNING - No index fields defined. Index field values will not be checked when writing to tables.
2026-09-17 17:28:54,670 - INFO - Successfully instantiated table ANALYSIS_METADATA
2026-09-17 17:28:54,670 - WARNING - No index fields found in ATTRIBUTE_DESIGN. Initializing index fields.
2026-09-17 17:28:54,671 - INFO - Successfully instantiated table ATTRIBUTE_DESIGN
2026-09-17 17:28:54,671 - WARNING - No index fields found in ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES. Initializing index fields.
2026-09-17 17:28:54,671 - INFO - Successfully instantiated table ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES
2026-09-17 17:28:54,671 - WARNING - No index fi

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Precompiling NemoMod...
Info Given NemoMod was explicitly requested, output will be shown live 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
   1255.9 ms  ? NemoMod
[ Info: Precompiling NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72] 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
2026-09-17 17:29:38,829 - INFO - Successfully initialized JuMP optimizer from solver module HiGHS.
2026-09-17 17:29:38,842 - INFO - Successfully initialized SISEPUEDEModels.
2026-09-17 17:29:38,849 - INFO - Table ANALYSIS_METADATA successfully written to /Users/fabianfuentes/miniconda3/envs/ssp_morocco_env/li

In [29]:
not(energy_model_flag)

False

In [ ]:
# This runs the model, make sure you edit key_stretegy with the strategy ids you want to execute include baseline (0)
dict_scens = {
    ssp.key_design: [0],
    ssp.key_future: [0],
    ssp.key_strategy: strategies_to_run,
}

ssp.project_scenarios(
    dict_scens,
    save_inputs = True,
    include_electricity_in_energy = energy_model_flag,
    # dict_optimizer_attributes = {"user_bound_scale": -7, }
)

2026-09-17 17:29:40,105 - INFO - 
***	STARTING REGION morocco	***

2026-09-17 17:29:41,119 - INFO - Trying run primary_id = 0 in region morocco
2026-09-17 17:29:41,119 - INFO - Running AFOLU model
2026-09-17 17:29:41,317 - INFO - AFOLU model run successfully completed
2026-09-17 17:29:41,317 - INFO - Running CircularEconomy model
2026-09-17 17:29:41,337 - INFO - CircularEconomy model run successfully completed
2026-09-17 17:29:41,337 - INFO - Running IPPU model
2026-09-17 17:29:41,369 - INFO - IPPU model run successfully completed
2026-09-17 17:29:41,369 - INFO - Running Energy model (EnergyConsumption without Fugitive Emissions)
2026-09-17 17:29:41,378 - DEBUG - Missing elasticity information found in 'project_energy_consumption_by_fuel_from_effvars': using specified future demands.
2026-09-17 17:29:41,415 - INFO - EnergyConsumption without Fugitive Emissions model run successfully completed
2026-09-17 17:29:41,416 - INFO - Running Energy model (Electricity and Fuel Production: trying

2026-17-Sep 17:29:41.769 Opened SQLite database at /Users/fabianfuentes/miniconda3/envs/ssp_morocco_env/lib/python3.11/site-packages/sisepuede/tmp/nemomod_intermediate_database.sqlite.
2026-17-Sep 17:29:41.787 Added NEMO structure to SQLite database at /Users/fabianfuentes/miniconda3/envs/ssp_morocco_env/lib/python3.11/site-packages/sisepuede/tmp/nemomod_intermediate_database.sqlite.
2026-17-Sep 17:29:56.215 Started modeling scenario. NEMO version = 2.2.0, solver = HiGHS.


2026-09-17 17:30:31,269 - INFO - NemoMod ran successfully with the following status: OPTIMAL
2026-09-17 17:30:31,275 - INFO - EnergyProduction model run successfully completed
2026-09-17 17:30:31,275 - INFO - Running Energy (Fugitive Emissions) and adjusting conversion emissions for biomass.
2026-09-17 17:30:31,309 - INFO - Fugitive Emissions from Energy model run successfully completed
2026-09-17 17:30:31,309 - INFO - Appending Socioeconomic outputs
2026-09-17 17:30:31,314 - INFO - Socioeconomic outputs successfully appended.
2026-09-17 17:30:31,316 - INFO - Model run for primary_id = 0 successfully completed in 50.2 seconds (n_tries = 1).
2026-09-17 17:30:31,329 - INFO - Trying run primary_id = 73073 in region morocco
2026-09-17 17:30:31,329 - INFO - Running AFOLU model


2026-17-Sep 17:29:56.663 Started optimizing following years: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011].
2026-17-Sep 17:30:10.064 Finished optimizing following years: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011].
2026-17-Sep 17:30:10.131 Started optimizing following years: [1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035].
2026-17-Sep 17:30:31.154 Finished optimizing following years: [1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035].
set ['emission_co2e_co2_entc_bmass_processing_and_refinement_fp_hydrogen_gasification'] = 0 in energy production. FIX WITH NEW fp_hydrogen_gasification_biomass TECH.


2026-09-17 17:30:31,539 - INFO - AFOLU model run successfully completed
2026-09-17 17:30:31,539 - INFO - Running CircularEconomy model
2026-09-17 17:30:31,559 - INFO - CircularEconomy model run successfully completed
2026-09-17 17:30:31,559 - INFO - Running IPPU model
2026-09-17 17:30:31,591 - INFO - IPPU model run successfully completed
2026-09-17 17:30:31,591 - INFO - Running Energy model (EnergyConsumption without Fugitive Emissions)
2026-09-17 17:30:31,600 - DEBUG - Missing elasticity information found in 'project_energy_consumption_by_fuel_from_effvars': using specified future demands.
2026-09-17 17:30:31,638 - INFO - EnergyConsumption without Fugitive Emissions model run successfully completed
2026-09-17 17:30:31,638 - INFO - Running Energy model (Electricity and Fuel Production: trying to call Julia)


...

2026-09-17 18:02:01,386 - INFO - AFOLU model run successfully completed
2026-09-17 18:02:01,387 - INFO - Running CircularEconomy model
2026-09-17 18:02:01,406 - INFO - CircularEconomy model run successfully completed
2026-09-17 18:02:01,407 - INFO - Running IPPU model
2026-09-17 18:02:01,439 - INFO - IPPU model run successfully completed
2026-09-17 18:02:01,439 - INFO - Running Energy model (EnergyConsumption without Fugitive Emissions)
2026-09-17 18:02:01,448 - DEBUG - Missing elasticity information found in 'project_energy_consumption_by_fuel_from_effvars': using specified future demands.
2026-09-17 18:02:01,486 - INFO - EnergyConsumption without Fugitive Emissions model run successfully completed
2026-09-17 18:02:01,487 - INFO - Running Energy model (Electricity and Fuel Production: trying to call Julia)


2026-17-Sep 18:02:02.011 Started modeling scenario. NEMO version = 2.2.0, solver = HiGHS.
2026-17-Sep 18:02:02.058 Started optimizing following years: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011].


2026-09-17 18:02:51,095 - INFO - NemoMod ran successfully with the following status: OPTIMAL
2026-09-17 18:02:51,101 - INFO - EnergyProduction model run successfully completed
2026-09-17 18:02:51,102 - INFO - Running Energy (Fugitive Emissions) and adjusting conversion emissions for biomass.
2026-09-17 18:02:51,143 - INFO - Fugitive Emissions from Energy model run successfully completed
2026-09-17 18:02:51,143 - INFO - Appending Socioeconomic outputs
2026-09-17 18:02:51,147 - INFO - Socioeconomic outputs successfully appended.
2026-09-17 18:02:51,149 - INFO - Model run for primary_id = 156156 successfully completed in 49.97 seconds (n_tries = 1).
2026-09-17 18:02:51,151 - INFO - Trying run primary_id = 157157 in region morocco
2026-09-17 18:02:51,152 - INFO - Running AFOLU model


2026-17-Sep 18:02:08.656 Finished optimizing following years: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011].
2026-17-Sep 18:02:08.710 Started optimizing following years: [1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035].
2026-17-Sep 18:02:51.019 Finished optimizing following years: [1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035].
set ['emission_co2e_co2_entc_bmass_processing_and_refinement_fp_hydrogen_gasification'] = 0 in energy production. FIX WITH NEW fp_hydrogen_gasification_biomass TECH.


2026-09-17 18:02:51,358 - INFO - AFOLU model run successfully completed
2026-09-17 18:02:51,359 - INFO - Running CircularEconomy model
2026-09-17 18:02:51,379 - INFO - CircularEconomy model run successfully completed
2026-09-17 18:02:51,379 - INFO - Running IPPU model
2026-09-17 18:02:51,411 - INFO - IPPU model run successfully completed
2026-09-17 18:02:51,412 - INFO - Running Energy model (EnergyConsumption without Fugitive Emissions)
2026-09-17 18:02:51,420 - DEBUG - Missing elasticity information found in 'project_energy_consumption_by_fuel_from_effvars': using specified future demands.
2026-09-17 18:02:51,459 - INFO - EnergyConsumption without Fugitive Emissions model run successfully completed
2026-09-17 18:02:51,459 - INFO - Running Energy model (Electricity and Fuel Production: trying to call Julia)


## Read simulations and check outputs

In [ ]:
# Read input and output files
df_out = ssp.read_output(None)
df_in = ssp.read_input(None)

In [ ]:
def plot_field_stack(
    df,
    fields,
    dict_format,
    time_col="time_period",
    primary_id=0,
    figsize=(18, 8),
    legend_loc='upper right',
    legend_bbox=(1.1, 1),
    ylabel="MT Emissions CO2e",
    xlabel="Time Period",
    title=None,
):
    """
    Plots a stack plot of the selected fields for a given primary_id.

    Args:
        df (pd.DataFrame): DataFrame containing output data.
        fields (list): List of column names to plot.
        dict_format (dict): Formatting dictionary for colors.
        time_col (str): Name of the time column.
        primary_id (int): Value of primary_id to filter.
        figsize (tuple): Figure size.
        legend_loc (str): Legend location.
        legend_bbox (tuple): Legend bbox_to_anchor.
        ylabel (str): Y-axis label.
        xlabel (str): X-axis label.
        title (str): Plot title.
    """
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)

    df_plot = df[df[ssp.key_primary].isin([primary_id])]

    fig, ax = spu.plot_stack(
        df_plot,
        fields,
        dict_formatting=dict_format,
        field_x=time_col,
        figtuple=(fig, ax),
    )

    ax.legend(loc=legend_loc, bbox_to_anchor=legend_bbox, title="Fields")
    plt.show()

In [ ]:
# Define the fields to plot and the formatting dictionary
subsector_emission_fields = matt.get_all_subsector_emission_total_fields()

dict_format = dict(
    (k, {"color": v}) for (k, v) in
    matt.get_subsector_color_map().items()
)

In [ ]:
primary_ids_to_plot = df_out[ssp.key_primary].unique()

In [ ]:
# Plot the emissions stack for the primary_id 0 (which is the baseline)
for primary_id in primary_ids_to_plot:

    plot_field_stack(
        df_out,
        subsector_emission_fields,
        dict_format,
        primary_id=primary_id,
        title=f"Emissions Stack Plot for Primary ID {primary_id}"
    )

# Export Wide File

In [ ]:
all_primaries = sorted(list(df_out[ssp.key_primary].unique()))

# build if unable to simply read the data frame
if df_in is None:
    df_in = []
     
    for region in ssp.regions:
        for primary in all_primaries: 
            df_in_filt = ssp.generate_scenario_database_from_primary_key(primary)
            df_in.append(df_in_filt.get(region))
    
    df_in = pd.concat(df_in, axis = 0).reset_index(drop = True)




df_export = pd.merge(
    df_out,
    df_in,
    how = "left",
)



# check output directory 
dir_pkg = os.path.join(
    ssp.file_struct.dir_out, 
    f"sisepuede_summary_results_run_{ssp.id_fs_safe}"
)
os.makedirs(dir_pkg) if not os.path.exists(dir_pkg) else None


for tab in ["ATTRIBUTE_STRATEGY"]:
    table_df = ssp.database.db.read_table(tab)
    if table_df is not None:
        table_df.to_csv(
            os.path.join(dir_pkg, f"{tab}.csv"),
            index=None,
            encoding="UTF-8"
        )
    else:
        print(f"Warning: Table {tab} returned None.")


df_primary = (
    ssp
    .odpt_primary
    .get_indexing_dataframe(
        sorted(list(df_out[ssp.key_primary].unique()))
    )
)
    
df_primary.to_csv(
    os.path.join(dir_pkg, f"ATTRIBUTE_PRIMARY.csv"),
    index = None,
    encoding = "UTF-8"
)

df_export.to_csv(
    os.path.join(dir_pkg, f"sisepuede_results_{ssp.id_fs_safe}_WIDE_INPUTS_OUTPUTS.csv"),
    index = None,
    encoding = "UTF-8"
)

In [ ]:
# Getting the directory where the outputs are stored
dir_pkg

In [ ]:
RUN_ID_OUTPUT_DIR_PATH = os.path.join(
    RUN_OUTPUT_DIR_PATH, 
    f"sisepuede_results_{ssp.id_fs_safe}"
)

os.makedirs(RUN_ID_OUTPUT_DIR_PATH, exist_ok=True)

df_primary.to_csv(
    os.path.join(RUN_ID_OUTPUT_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"),
    index = None,
    encoding = "UTF-8"
)

df_export.to_csv(
    os.path.join(RUN_ID_OUTPUT_DIR_PATH, f"sisepuede_results_{ssp.id_fs_safe}_WIDE_INPUTS_OUTPUTS.csv"),
    index = None,
    encoding = "UTF-8"
)

for tab in ["ATTRIBUTE_STRATEGY"]:
    table_df = ssp.database.db.read_table(tab)
    if table_df is not None:
        table_df.to_csv(
            os.path.join(RUN_ID_OUTPUT_DIR_PATH, f"{tab}.csv"),
            index=None,
            encoding="UTF-8"
        )
    else:
        logger.warning(f"Warning: Table {tab} returned None.")

In [ ]:
RUN_ID_OUTPUT_DIR_PATH